# Tensor simulation in FlagQuantum

Adapted from Qualtran v0.7.0 `qualtran/simulation/tensor.ipynb`, commit `bcb17bb25360ed68ce3bc084645db3e929799389`, Git blob `d91373e7d8eb21c48a0b870d99d1bb2973971861`.
Copyright 2023 Google LLC. SPDX-License-Identifier: Apache-2.0.

This notebook follows all 18 original code-cell workflows using FlagQuantum-owned IR, tensor suppliers, contraction and drawing. It imports no reference package. Dense examples here are bounded to seven output bits. It demonstrates tensor inspection; users can continue to append physical operations with `circuit.append(op)` as covered by the component examples.

In [ ]:
from functools import partial

import torch

from examples.algorithms.bell_tensor_workflow import (
    bell_program,
    bell_providers,
    contract_bell_program,
)
from examples.algorithms.factorized_controlled_not import (
    controlled_not_chain,
    factorized_cnot_supply,
)
from examples.algorithms.tensor_network_inspection import inspection_example
from flagquantum.compiler.registers.component_graph import iter_component_calls
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.tensor_network.component_tensor_nodes import (
    component_tensor_nodes_with_labels,
)
from flagquantum.simulation.tensor_network.register_tensor_binding import (
    contract_register_tensors,
)

like = torch.empty((), dtype=torch.complex128)
cnot_nodes, cnot_names = inspection_example()
# Native outer names retain register direction. Bind the matrix through its signature.

cnot_record = controlled_not_chain(1)
cnot_order = tuple(i for i, _, _ in iter_component_calls(cnot_record, "cnot_chain"))
cnot_nodes, cnot_in, cnot_out = component_tensor_nodes_with_labels(
    cnot_record,
    cnot_order,
    partial(factorized_cnot_supply, like=like),
    like,
    "cnot_chain",
)
cnot_matrix = contract_register_tensors(
    cnot_nodes, Signature(cnot_record.functions[0].registers or ()), cnot_in, cnot_out
)
cnot_matrix.real


States are column vectors and effects are covectors. The plus state is H applied to zero; the zero effect projects onto zero without a physical reset or measurement.

In [ ]:
from flagquantum.core.ir.program.registers import Register, RegisterSide, RegisterType
from flagquantum.simulation.tensor_network.register_basis_vectors import (
    register_basis_vector_nodes,
)

prepared = bell_program(prepared=True)
plus_function = next(f for f in prepared.functions if f.name == "plus")
plus_state = (
    bell_providers(prepared, like)
    .supply(plus_function, {}, {"val": ((0,),)}, lambda: 1)[0]
    .tensor
)
zero_effect = register_basis_vector_nodes(
    Signature((Register("q", RegisterType("bit", 1), side=RegisterSide.LEFT),)),
    {"q": ((0,),)},
    None,
    like,
    state=False,
)[0].tensor
print("|+>", plus_state)
print("<0|", zero_effect)


A two-control And has four input basis states and eight output basis states. Its forward tensor is an (8, 4) isometry, including the owned target result.

In [ ]:
from examples.algorithms.conjunction_tensors import (
    conjunction_program,
    conjunction_providers,
)
from flagquantum.simulation.tensor_network.component_dense import (
    contract_component_tensors,
)

and_record = conjunction_program((1, 1))
and_order = tuple(i for i, _, _ in iter_component_calls(and_record, "multi_and"))
and_matrix = contract_component_tensors(
    and_record, and_order, conjunction_providers((1, 1), like).supply, like, "multi_and"
)
and_matrix.shape


For n input bits and n output bits a dense unitary has shape (2**n, 2**n). The source's prose describes this as (n, n); the native text corrects the dimensional wording.

## Interface

Native component tensor wiring and dense contraction consume the same register signature and explicit leaf suppliers. The independent physical unitary owner provides a second CNOT check.

In [ ]:
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.simulation.unitary import get_unitary

cnot_equal = torch.equal(
    cnot_matrix,
    get_unitary(CircuitIR(2, (Instruction("cx", (0, 1)),)), dtype=like.dtype),
)
assert cnot_equal
cnot_equal


## Additional functionality

The four-control conjunction is a ladder of three And leaves. Logical prefix outputs (`junk`) remain live and owned by the caller; they are not clean scratch at the forward exit. Shared component IR also feeds the register score below.

In [ ]:
from dataclasses import replace

from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_score import RegisterStep
from flagquantum.drawer.register_score_data import (
    PortSide,
    SymbolClass,
    _port_symbol,
    register_score_data,
)
from flagquantum.drawer.register_score_mpl import draw_register_score


def compact_symbols(
    step: RegisterStep, register: str | None, side: PortSide | None
) -> tuple[SymbolClass, dict[str, str | int | bool]]:
    """Shorten display labels without altering graph identities or placement."""
    if register is None:
        return "Text", {
            "text": "zero" if step.name.startswith("__fq_zero") else step.name,
            "fontsize": 10,
        }
    if step.name.startswith("and_"):
        return _port_symbol(
            replace(step, gate="and", control_values=(1, 1)), register, side
        )
    if step.name == "cnot":
        return _port_symbol(replace(step, gate="cx"), register, side)
    symbol: SymbolClass = "LarrowTextBox" if side == "RIGHT" else "TextBox"
    return symbol, {
        "text": (
            "0" if step.name.startswith("__fq_zero") else register.rsplit(":", 1)[-1]
        )
    }


values = (1,) * 4
multi_record = conjunction_program(values)
multi_providers = conjunction_providers(values, like)
multi_order = tuple(i for i, _, _ in iter_component_calls(multi_record, "multi_and"))
# All three logical And calls remain visible; their physical definitions are unchanged.
multi_score = register_score_data(
    component_register_score(multi_record, "multi_and"), wire_symbol=compact_symbols
)
figure, axes = draw_register_score(multi_score)


Draw the same numerical leaves as an incidence graph. Register endpoints and internal edges match the source topology. Native Matplotlib presentation replaces Graphviz and Quimb layout; spectral coordinates and pixel identity are not promised.

In [ ]:
from flagquantum.drawer.tensor_network import (
    draw_tensor_network,
    tensor_index_occurrences,
)
from flagquantum.simulation.tensor_network.component_tensor_nodes import (
    component_tensor_nodes,
)
from flagquantum.simulation.tensor_network.register_tensor_indices import (
    register_tensor_outer_indices,
)

multi_nodes, multi_in, multi_out = component_tensor_nodes(
    multi_record, multi_order, multi_providers.supply, like, "multi_and"
)
multi_signature = Signature(multi_record.functions[0].registers or ())
multi_names = {
    k: v
    for k, v in register_tensor_outer_indices(
        multi_signature, multi_in, multi_out, friendly=True
    ).items()
    if isinstance(v, str)
}
figure, axes = draw_tensor_network(multi_nodes, index_names=multi_names)


Inspect the existing native shape-only contraction profile. Native union-dimension cost is not opt_einsum FLOPs: for this example 4,608 native units and 9,216 source FLOPs correspond to the same path, with 2 steps and 2,048 peak elements. The 32,768-byte complex128 dense output is not process RAM. This is not a complete Quimb optimizer interface.

In [ ]:
from examples.algorithms.conjunction_contraction import contract_conjunction_with_info

multi_matrix, contraction_info = contract_conjunction_with_info(values, like)
print("Native union-dimension estimate:", contraction_info.estimated_cost)
print("Peak tensor elements:", contraction_info.peak_size)
print("Contraction steps:", contraction_info.n_steps)


Native inner indices are graph-owned integers. The occurrence map gives (tensor ordinal, axis) endpoints; source indices carry (Connection, bit) objects instead. Friendly names annotate external ports. Split/join grouping preserves individual bit axes instead of fusing them. No foreign connection object or duplicate graph is created.

In [ ]:
occurrences = tensor_index_occurrences(multi_nodes)
example_inner_index = next(
    label for label, endpoints in occurrences.items() if len(endpoints) > 1
)
print("Native edge:", example_inner_index)
print("(tensor ordinal, axis) endpoints:", occurrences[example_inner_index])


Explicit logical suppliers take precedence over physical bodies. Full and partial selection produce the same map here. Do not flatten measurement-assisted physical erasure into a pure state tensor. Selection reuses the existing Compiler owner.

In [ ]:
# Logical And suppliers take precedence over their physical decompositions.
multi_partial = contract_component_tensors(
    multi_record, multi_order, multi_providers.supply, like, "multi_and"
)
torch.testing.assert_close(multi_partial, multi_matrix, rtol=0, atol=0)


In [ ]:
from examples.algorithms.conjunction_contraction import select_conjunction_instances

custom_flat, selected_instances = select_conjunction_instances(
    values, like, keep_instance=2
)
custom_order = tuple(i for i, _, _ in iter_component_calls(custom_flat, "tensor_root"))
custom_nodes, _, _ = component_tensor_nodes(
    custom_flat, custom_order, multi_providers.supply, like, "tensor_root"
)
print("Retained logical instances:", selected_instances)
len(custom_nodes)


## Custom tensors

Bind a complex128 CNOT matrix through the native signature. Output axes precede input axes, in ctrl,target order. This replaces a leaf representation without replacing the consumer or numerical graph.

In [ ]:
from flagquantum.simulation.tensor_network.register_tensor import (
    register_tensor_from_unitary,
)
from flagquantum.simulation.tensor_network.register_tensor_binding import (
    bind_register_tensor,
)

# The custom provider uses output ctrl,target then input ctrl,target axes.
custom_matrix = torch.tensor(
    [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=torch.complex128
)
custom_signature = Signature(
    (
        Register("ctrl", RegisterType("bit", 1)),
        Register("target", RegisterType("bit", 1)),
    )
)
custom_node = bind_register_tensor(
    register_tensor_from_unitary(custom_matrix, custom_signature),
    custom_signature,
    cnot_in,
    cnot_out,
)


In [ ]:
custom_result = contract_register_tensors(
    (custom_node,), custom_signature, cnot_in, cnot_out
)
torch.testing.assert_close(custom_result, cnot_matrix, rtol=0, atol=0)
custom_result


## Decomposition fallback

Bell preparation composes PlusState, ZeroState and CNOT. The root has no custom tensor. Expansion resolves its known leaves; requesting the unexpanded root must fail with a missing-provider diagnostic.

In [ ]:
# Reuse the real PlusState/ZeroState/CNOT composition, with owned output ports.
bell_prepared = bell_program(prepared=True)


Contract the expanded Bell preparation. The two output bits are logical results, with no extra scratch. The complete complex vector is compared, including phase.

In [ ]:
bell_state = contract_bell_program(bell_prepared, like)
print(bell_state)


The same root without expansion intentionally fails. This is an explicit capability boundary, not a silent simulator substitution.

In [ ]:
from flagquantum.compiler.registers.component_tensor_expansion import (
    expand_component_root_for_tensor,
)

# Deliberately retain the root itself, bypassing normal supplier-aware expansion.
unexpanded, _ = expand_component_root_for_tensor(
    bell_prepared, lambda _: True, "bell", full_flatten=False
)
unexpanded_order = tuple(
    i for i, _, _ in iter_component_calls(unexpanded, "tensor_root")
)
unflattened_refused = False
try:
    component_tensor_nodes_with_labels(
        unexpanded,
        unexpanded_order,
        bell_providers(bell_prepared, like).supply,
        like,
        "tensor_root",
    )
except ValueError as error:
    if "no tensor provider" not in str(error):
        raise
    unflattened_refused = True
    print("Expected missing Bell root provider:", error)
assert unflattened_refused


## Properties and relations

Factorized CNOT uses the original COPY and XOR rank-three tensor data. Their internal bond is a numerical edge, not a physical ancilla.

Inspect every leaf and draw the native topology. Tag names and tensor data correspond to source COPY/XOR, while figure layout follows the native drawer.

In [ ]:
figure, axes = draw_tensor_network(
    cnot_nodes,
    index_names=cnot_names,
    node_colors={"COPY": "#b6d9ff", "XOR": "#ffd39e"},
)
for node in cnot_nodes:
    print(node.name)
    print(node.tensor)


A circuit with two input bits represents a 4 by 4 unitary. H then CNOT uses one H and one CX physically. Source model estimates are separate: two Clifford operations for the unitary versus one for the prepared Bell composition (state preparation has its own model convention). Neither has T gates or measurements.

In [ ]:
bell_circuit = bell_program(prepared=False)
bell_score = register_score_data(
    component_register_score(bell_circuit, "bell"), wire_symbol=compact_symbols
)
figure, axes = draw_register_score(bell_score)
bell_unitary = contract_bell_program(bell_circuit, like)
print(bell_unitary)


Initialize both inputs to zero using the existing register transformation, then contract the resulting state vector. This agrees with direct Bell preparation. Physical append/inverse, phase, spectator, gradients and all small erasure-branch checks remain in the linked component regression; this notebook does not introduce new gate implementations.

In [ ]:
from flagquantum.compiler.registers.component_zero_initialization import (
    initialize_component_from_zero,
)

bell_initialized = initialize_component_from_zero(bell_circuit, "bell")
initialized_score = register_score_data(
    component_register_score(bell_initialized, "from_zero"), wire_symbol=compact_symbols
)
figure, axes = draw_register_score(initialized_score)
bell_from_zero = contract_bell_program(bell_initialized, like, entry="from_zero")
torch.testing.assert_close(bell_from_zero, bell_state, rtol=0, atol=0)
print(bell_from_zero)
